In [45]:
#loading libraries
import pandas as pd
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side, GradientFill
from openpyxl.utils import get_column_letter #converts column with into letter
from openpyxl.chart import BarChart, LineChart, Reference
from openpyxl.formatting.rule import ColorScaleRule, DataBarRule #conditional formatting rules


In [46]:
#loading csv
df= pd.read_csv('Mobile Payments.csv')
df.head()

,Year,Month,Active Agents,Total Registered Mobile Money Accounts (Millions),Total Agent Cash in Cash Out (Volume Million),Total Agent Cash in Cash Out (Value KSh billions)
0,2026,April,548010,92.84,212.29,680.99
1,2026,March,621389,91.39,217.86,693.37
2,2026,February,507383,91.32,196.15,633.35
3,2026,January,474786,90.41,215.08,699.64
4,2025,December,473536,89.46,217.58,722.53


In [47]:
#cleaning the data
month_order=['January','February','March','April','May','June','July','August','September','October','November','December']
#telling pandas months have custom sort order, not alphabetically
df['Month'] = pd.Categorical(df['Month'], categories= month_order, ordered=True)

df=df.sort_values(['Year','Month']).reset_index(drop=True)
#adding new period column
df['Period'] = df['Month'].astype(str) + " " + df['Year'].astype(str)
df.head(5)

,Year,Month,Active Agents,Total Registered Mobile Money Accounts (Millions),Total Agent Cash in Cash Out (Volume Million),Total Agent Cash in Cash Out (Value KSh billions),Period
0,2007,March,307,0.020992,0.021714,0.064391,March 2007
1,2007,April,362,0.054944,0.070000,0.220896,April 2007
2,2007,May,447,0.107733,0.150000,0.483709,May 2007
3,2007,June,527,0.175652,0.233661,0.720102,June 2007
4,2007,July,681,0.268499,0.354298,1.065370,July 2007


In [48]:
#renaming columns
df.rename(columns ={'Total Registered Mobile Money Accounts (Millions)': 'Registered Accounts(M)',
                   'Total Agent Cash in Cash Out (Volume Million)':'Transaction Volume(M)','Total Agent Cash in Cash Out (Value KSh billions)':'Transaction Value(B)' },
         inplace=True)
df.head(3)

,Year,Month,Active Agents,Registered Accounts(M),Transaction Volume(M),Transaction Value(B),Period
0,2007,March,307,0.020992,0.021714,0.064391,March 2007
1,2007,April,362,0.054944,0.070000,0.220896,April 2007
2,2007,May,447,0.107733,0.150000,0.483709,May 2007


In [49]:
#annual data summary with pandas
annual = df.groupby('Year',as_index=False).agg(
    Avg_Active_Agents =('Active Agents','mean'),
    Total_Volume_M = ('Transaction Volume(M)','sum'),
    Total_Value_Ksh_B=('Transaction Value(B)','sum'),
    Months_recorded = ('Month','count')
    
).round(2)
annual['Avg_Active_Agents']= annual['Avg_Active_Agents'].round(0).astype(int)
#annual yearly growth
annual['Yearly Value Growth %']= (annual['Total_Value_Ksh_B'].pct_change()).round(1)

annual.head()

,Year,Avg_Active_Agents,Total_Volume_M,Total_Value_Ksh_B,Months_recorded,Yearly Value Growth %
0,2007,826,5.47,16.32,10,NaN
1,2008,3521,62.74,166.57,12,9.2
2,2009,16570,193.50,473.41,12,1.8
3,2010,32270,311.05,732.22,12,0.5
4,2011,42115,433.00,1169.15,12,0.6


In [50]:
#define colors
DARK_BLUE  = "1F4E79"
MID_BLUE   = "2E75B6"
LIGHT_BLUE = "D6E4F0"
ACCENT     = "00B0F0"
WHITE      = "FFFFFF"
DARK_GRAY  = "404040"
ALT_ROW    = "F2F7FC"

#style helpers functions- ready made style objects
def hdr_font(size=11): return Font(bold=True, color=WHITE, name='Calibri',size=size)
def body_font(size=10): return Font(name='Calibri',size=size,color=DARK_GRAY)
def title_font(): return Font(bold=True,name='Calibri',size=14,color=DARK_BLUE)
def kpi_val_font(): return Font(bold=True, name='Calibri',size=18,color=MID_BLUE)
def kpi_label_font(): return Font(name='Calibri',size=9,color='808080')
def kpi_sub_font():    return Font(name="Calibri", size=8, color="808080", italic=True)
def kpi_label_fill():  return PatternFill("solid", fgColor=MID_BLUE)
def kpi_val_fill():  return PatternFill("solid", fgColor=LIGHT_BLUE)    

def hdr_fill(color= DARK_BLUE): return PatternFill('solid',fgColor=color)
def alt_fill(): return PatternFill('solid',fgColor=ALT_ROW)

THIN = Side(style='thin',color='D0D7DF')
MED = Side(style='medium',color=MID_BLUE)
def cell_border(): return Border(left =THIN, right=THIN, top=THIN, bottom=THIN)
def hdr_border(): return Border(left=MED, right=MED, top=MED, bottom=MED)

def center(): return Alignment(horizontal='center', vertical='center')
def left(): return Alignment(horizontal='left',vertical='center')
def right(): return Alignment(horizontal='right', vertical='center')

In [51]:
#auto width
def auto_width(ws,extra=4, max_w=35):
    for col in ws.columns:
        best = max((len(str(c.value or "")) for c in col), default=8)
        ws.column_dimensions[get_column_letter(col[0].column)].width = min(best + extra, max_w)

In [52]:
#reusable function taking in worksheet(ws), headers, row, fill_color
def write_header_row(ws,headers, row=1, fill_color=DARK_BLUE):
    #enumerate to loop through headers lists and give counter-ci and value h
    for ci, h in enumerate(headers,1):
        c= ws.cell(row=row, column=ci, value=h)
        #applying styles
        c.font=hdr_font()
        c.fill=hdr_fill(fill_color)
        c.alignment=center()
        c.border= hdr_border()

In [53]:
# write data to rows
#row 1 is always header, number_cols that need decimal formatting,pct_cols that need formatting
def write_data_rows(ws, dataframe, start_row=2, number_cols=None,pct_cols=None,int_cols=None):
    #itertuples faster than iterrows-create full series object per row- moves row by row
    for ri,row in enumerate(dataframe.itertuples(index=False),start_row):
        #alt colors rows
        use_alt=(ri%2 == 0)
        #one iteration per cell- cell by cell 
        for ci, val in enumerate(row,1):
            c=ws.cell(row=ri,column=ci,value=val)
            c.font=body_font()
            c.border =cell_border()
            c.alignment = center()
            if use_alt:
                c.fill=alt_fill()
            if ci in number_cols:
                c.number_format='#,##0.00'
            elif ci in pct_cols:
                c.number_format='+0.0%;-.%;0.0%'
            elif ci in int_cols:
                c.number_format ='#,##0'
                

## Creating Workbooks and Sheets

In [54]:
#creating workbooks and sheets
wb= Workbook()
ws_exec=wb.active #first sheet already exists
ws_exec.title ='Executive Summary'
ws_exec.sheet_view.showGridLines = False
ws_exec.row_dimensions[1].height =8


### Executive Summary Sheet

In [55]:
#Executive summary, title banner and kpi cards
ws_exec.merge_cells('B2:I2')
t= ws_exec['B2']
t.value ='Kenya Mobile Money -  M-PESA Growth Report (2007 - 2026)'
t.font = Font(bold=True, name='Calibri', size=16, color=WHITE)
t.fill = PatternFill('solid',fgColor=DARK_BLUE)
t.alignment = center()
ws_exec.row_dimensions[2].height = 36

In [56]:
ws_exec.merge_cells('B3:I3')
sub= ws_exec['B3']
sub.value = 'Source: CBK Mobile Payments Data | Generated automatically with python and openpyxl '
sub.font=Font(name='Calibri', size=9,color='808080', italic=True)
sub.fill = PatternFill('solid',fgColor=LIGHT_BLUE)
sub.alignment = center()
ws_exec.row_dimensions[3].height = 18

In [57]:
#kpis cards rows
latest= df.iloc[-1]
first= df.iloc[0]


kpis = [
    ('Peak Active Agents', f'{df['Active Agents'].max():,}',     "2026"),
    ('Registered Accounts', f'{df['Registered Accounts(M)'].max():.2f}M', "2026"),
    ('Annual Value(2025)',  f'KSh {annual[annual.Year==2025]['Total_Value_Ksh_B'].values[0]:,.1f}B', ''),
    ('Annual Volume(2025', f'{annual[annual.Year==2025]['Total_Volume_M'].values[0]:,.1f}M txns',''),
    ('Years of Data', '20 Years', '2007-2026'),
    ('Total Months', f'{len(df)} months', 'records')
        ]


In [58]:
# executing kpi rows in excel

ws_exec.row_dimensions[5].height =14
ws_exec.row_dimensions[6].height =40 #label row
ws_exec.row_dimensions[7].height =22 #kpi value
ws_exec.row_dimensions[8].height =14 # sub txt

for i, (lbl, val, sub_txt) in enumerate(kpis):
    col = i + 2 # col B - G
    #label
    lc= ws_exec.cell(row=6, column=col, value=lbl)
    lc.font=kpi_label_font()
    lc.fill= kpi_label_fill()
    lc.alignment=center()
    lc.border=cell_border()
    #value cell
    vc= ws_exec.cell(row=7, column=col, value=val)
    vc.font= kpi_val_font()
    vc.fill= kpi_val_fill()
    vc.alignment= center()
    vc.border=cell_border()
    #subtitle cell
    sbt = ws_exec.cell(row=8,column=col,value=sub_txt)
    sbt.font=kpi_sub_font()
    sbt.alignment=center()

for col in range(2,8):
    ws_exec.column_dimensions[get_column_letter(col)].width = 22

In [59]:
# Section headings, Key metric table

ws_exec.row_dimensions[10].height = 20
mh = ws_exec['B10']
mh.value ='Annual Summary'
mh.font = Font(bold=True, name='Calibri', size=12, color=WHITE)
mh.fill= PatternFill('solid',fgColor=MID_BLUE)
mh.alignment= left()
ws_exec.merge_cells('B10:G10')

ann_headers=['Year','Avg Active Agents', 'Avg Accounts(M)', 'Total Volume(M)',
             'Total Value(Ksh B)','Yearly Value Growth %', 'Months Recorded']
write_header_row(ws_exec, ann_headers, row=11, fill_color=DARK_BLUE)
write_data_rows(ws_exec, annual,start_row=12, number_cols=[3,4], pct_cols=[5],int_cols=[2])


In [60]:
# color scale yearly growth column
last_ann_row = 11 + len(annual) # last row where conditional formatting ends
ws_exec.conditional_formatting.add(
    f"F12:F{last_ann_row}",
    ColorScaleRule(
        start_type='min', start_color='F8696B',
        mid_type='num', mid_value=0, mid_color='FFEB84', # 0 is mid point -ve gets red and +ve gets green
        end_type='max', end_color='63BE7B'
        
    )
)
for col in range(1,8):
    ws_exec.column_dimensions[get_column_letter(col)].width = 22
    ws_exec.column_dimensions['A'].width=3


### Sheet 2 Raw Data

In [61]:
#Sheet 2 raw data
ws_raw = wb.create_sheet('Raw Data')
ws_raw.freeze_panes='A2' # row 1 remains frozen,not cols, row 2 onwards scollable

#cols
raw_cols =['Year','Month','Active Agents','Registered Accounts(M)','Transaction Volume(M)',
          'Transaction Value(B)']
write_header_row(ws_raw, raw_cols, row=1)
write_data_rows(ws_raw, df[raw_cols], start_row=2, number_cols=[4,5,6],pct_cols=[], int_cols=[3])

#Data bars on Transaction Value column(f6)
ws_raw.conditional_formatting.add(
    f"F2:F{1+len(df)}", DataBarRule(start_type='min', end_type='max',color=ACCENT, showValue=True)
 )

#color scales on Active Agents(C#)
ws_raw.conditional_formatting.add(
    f'C2:C{1+len(df)}', ColorScaleRule(start_type='min', start_color='FFF2CC', end_type='max',end_color='00B050')

)
auto_width(ws_raw)
ws_raw.row_dimensions[1].height=22

### Sheet 3- Annual Trends

In [62]:
ws_trend= wb.create_sheet('Annual Trends')
ws_trend.sheet_view.showGridLines= False

t2= ws_trend['A1']
t2.value='Annual Trends : Transaction Value & Volume'
t2.font = title_font()
ws_trend.row_dimensions[1].height = 28


In [66]:
#trend cols
trend_cols=['Year','Total Volume(M)','Total Value(ksh B)','Yearly Value Growth %']
write_header_row(ws_trend, trend_cols, row=3)
write_data_rows(ws_trend, annual[['Year','Total_Volume_M','Total_Value_Ksh_B','Yearly Value Growth %']],
               start_row=4,number_cols=[2,3],int_cols=[],pct_cols=[4])

In [68]:
#conditional format on growth col
last_tr = 3 + len(annual)
ws_trend.conditional_formatting.add(
    f'D4:D{last_tr}', ColorScaleRule(start_type='min', start_color='F8696B',
                                    mid_type='num',mid_value=0, mid_color='FFFF00',
                                    end_type='max', end_color='63BE7B')
)
auto_width(ws_trend, extra=6)

### Charts


In [ ]:
# Total value by Year bar chart